In [16]:
########
#Base parameters
########

lA = 2
lB = 3
eA = 4
eB = 3
p = (lA^(eA)) * (lB^(eB)) - 1 

In [17]:
F.<z> = GF(p^2, modulus=x^2+1)

In [19]:
E0 = EllipticCurve([F(1), F(0)])

In [20]:
########
#Setting up the respective generators
########
Gen1 = E0.gens()[0]
Gen2 = E0.gens()[1]

In [21]:
########
#Initiator random walk
########
def awalk(E,lA,lB,eA,eB,x,y):
    G = x*Gen1 + y*Gen2
    EE = E
    for i in range(0,eA):
        K = (lA^(eA-1-i))*G
        iso = EllipticCurveIsogeny(EE,K)
        EE = iso.codomain()
        G = iso(G)
        if (lB^(eB))G == EE(0,1,0):
                break
            
            
    for i in range(0,eB):
        K = (lB^(eB-1-i))*G
        iso = EllipticCurveIsogeny(EE,K)
        EE = iso.codomain()
        G = iso(G) 
        if G == EE(0,1,0):
                break
            
    return EE

In [22]:
########
#Public key generation. The last two input points are the other party's basis points. The first output is the image curve, the latter two are the the public basis point images.
########

def public(E,ll,e,m,n,P,Q,PP,QQ): 
    G = m*P + n*Q
    EE = E
    Pi = PP
    Qi = QQ
    
    for i in range(0,e):
            K = (ll^(e-1-i))*G
            iso = EllipticCurveIsogeny(EE,K)
            EE = iso.codomain()
            Pi = iso(Pi)
            Qi = iso(Qi)
            G = iso(G)
    
    return EE, Pi, Qi

In [23]:
########
#Private key generation. The input curve is the other party's image curve. The last two input points are the public points. 
########

def key(E,ll,e,m,n,P,Q): 
    G = m*P + n*Q
    EE = E
    
    for i in range(0,e):
        K = (ll^(e-1-i))*G
        iso = EllipticCurveIsogeny(EE,K)
        EE = iso.codomain()
        G = iso(G)
        
    return EE.j_invariant() 

In [8]:
########
#Setting up the exchange
########

In [36]:
########
#Initiator public key generation
########

def publicA(E,mA,nA): 
    Gen1A = lA^eA * E.gens()[0]
    Gen2A = lA^eA * E.gens()[1]
    Gen1B = lB^eB * E.gens()[0]
    Gen2B = lB^eB * E.gens()[1]
    return public(E,lA,eA,mA,nA,Gen1A,Gen2A,Gen1B,Gen2B)

In [37]:
########
#Responder public key generation
########

def publicB(E,mB,nB):
    Gen1A = lA^eA * E.gens()[0]
    Gen2A = lA^eA * E.gens()[1]
    Gen1B = lB^eB * E.gens()[0]
    Gen2B = lB^eB * E.gens()[1]
    return public(E,lB,eB,mB,nB,Gen1B,Gen2B,Gen1A,Gen2A)

In [30]:
########
#Initiator private key generation
########

def keyA(mA,nA,E,P,Q): #Where the outputs of the public key are E,P,Q respectively 
    return key(E,lA,eA,mA,nA,P,Q) 

In [31]:
########
#Responder private key generation
########

def keyB(mB,nB,E,P,Q):
    return key(E,lB,eB,mB,nB,P,Q) 

In [39]:
########
#Exchange testing
########

def test(E,mA,nA,mB,nB,x,y):
    E1 = awalk(E,lA,lB,eA,eB,x,y)
    pA = publicA(E1,mA,nA)
    pB = publicB(E1,mB,nB)
    testkeyA = keyA(mA,nA,pB[0],pB[1],pB[2])
    testkeyB = keyB(mB,nB,pA[0],pA[1],pA[2])
    if testkeyA == testkeyB:
        return "success", testkeyA
    else:
        return "something has gone wrong"

In [43]:
test(E0,3,4,5,6,7,11)

('success', 344*z + 190)